# Cotton Leaf Disease Detection - Model Training

Eight cells. Run them in order.

**Before you start**

1. **Runtime -> Change runtime type -> T4 GPU -> Save**
2. Have your `kaggle.json` ready. kaggle.com -> profile picture -> Settings ->
   API Tokens tab -> **Create Legacy API Key** at the bottom.
3. In Cell 1, paste your dataset slug into `DATASET`.

Expect 25 to 40 minutes on a GPU.

Every number and figure comes from your own run. Read the notes above each cell,
because those are the things you will be asked to explain.

---
## Cell 1 - Install, import and configure

Everything in one place: packages, imports, settings.

Two settings matter for your report.

- **`IMG = 224`.** Your report says 224 pixels, the old code used 256. This must
  match `app.py` too, or accuracy collapses at deployment for no visible reason.
- **`EXPECTED`.** Class folder names copied exactly from the Kaggle Data Explorer,
  including capitals and spaces.

In [ ]:
# ========================= INSTALL =========================
!pip install -q kaggle

# ========================= IMPORTS =========================
import os, re, json, random, shutil, hashlib
from collections import defaultdict, Counter

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

# ========================= SETTINGS =========================
DATASET = "PASTE/YOUR-SLUG-HERE"     # Kaggle page -> three dots -> Copy API command

EXPECTED = ["Bacterial Blight", "Curl Virus", "Healthy Leaf",
            "Herbicide Growth Damage", "Leaf Hopper Jassids",
            "Leaf Redding", "Leaf Variegation"]

IMG        = 224          # must match app.py
BATCH      = 32
EPOCHS     = 30           # EarlyStopping normally halts well before this
LR         = 1e-3
SEED       = 42
TRAIN_FRAC = 0.65         # validation 0.15, test 0.20

RAW, SPLIT_DIR = "data/raw", "data/split"
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# Folders of processed copies rather than real photographs. Training on these
# would repeat the exact fault this project is correcting.
SKIP_WORDS = ("augment", "hog", "black and white", "grayscale", "greyscale")

# Filename fragments marking an augmented copy of another photograph.
AUG_HINTS = ("rotation", "rotate", "zoom", "flip", "mirror", "contrast",
             "constract", "crop", "translation", "rotozoom", "pil_",
             "bright", "aug", "noise", "shear")

random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
plt.rcParams.update({"font.size": 10, "axes.grid": True, "grid.alpha": 0.3,
                     "figure.dpi": 120, "savefig.dpi": 200})
for d in ("figures", "results", "models"):
    os.makedirs(d, exist_ok=True)

# ========================= GPU CHECK =========================
gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPU:", gpus[0].name if gpus else "NONE - set Runtime to T4 GPU first")
print("Classes expected:", len(EXPECTED))

---
## Cell 2 - Sign in to Kaggle, download, keep only real photographs

Pick your `kaggle.json` when the **Choose Files** button appears.

Three jobs here.

1. Downloads straight to Colab's machine, so nothing touches your laptop.
2. **Deletes folders of processed copies.** Some datasets ship an
   `Augmented Dataset` beside an `Original Dataset`. Training on the augmented one
   recreates the fault being fixed, so it goes before anything reads it.
3. Finds your class folders, which Kaggle archives often bury several levels down.

A 403 means you need to open the dataset page in a browser once and accept its
terms. A 404 means the slug is wrong.

In [ ]:
# ------------- credentials -------------
if not os.path.exists("/root/.kaggle/kaggle.json"):
    print("Choose your kaggle.json file:")
    up = files.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    with open("/root/.kaggle/kaggle.json", "wb") as f:
        f.write(up["kaggle.json"])
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("Kaggle user:", json.load(open("/root/.kaggle/kaggle.json"))["username"])

# ------------- download -------------
os.makedirs(RAW, exist_ok=True)
if not any(os.scandir(RAW)):
    print("\nDownloading", DATASET, "...")
    rc = os.system("kaggle datasets download -d " + DATASET + " -p " + RAW + " --unzip")
    if rc != 0:
        raise SystemExit("Download failed. 403 = accept the dataset terms in a "
                         "browser first. 404 = the slug is wrong.")
else:
    print("\nAlready downloaded.")

# ------------- drop processed copies -------------
removed = []
for dirpath, dirnames, _ in os.walk(RAW, topdown=True):
    for d in list(dirnames):
        if any(w in d.lower() for w in SKIP_WORDS):
            shutil.rmtree(os.path.join(dirpath, d), ignore_errors=True)
            dirnames.remove(d)
            removed.append(d)
if removed:
    print("\nRemoved processed copies, keeping only real photographs:")
    for d in sorted(set(removed)):
        print("   ", d)

# ------------- locate class folders -------------
def find_class_level(root, expected):
    named, matched = [], []
    for dirpath, dirnames, _ in os.walk(root):
        hits = sum(1 for c in expected if c in set(dirnames))
        if hits >= 2:
            matched.append((hits, dirpath))
            if "original" in os.path.basename(dirpath).lower():
                named.append((hits, dirpath))
    pool = named or matched
    if not pool:
        return None
    pool.sort(key=lambda t: (-t[0], len(t[1])))
    return pool[0][1]

level = find_class_level(RAW, EXPECTED)

if level is None:
    print("\nCould not find the expected class folders. Tree below:")
    for dp, dn, fn in os.walk(RAW):
        depth = dp.replace(RAW, "").count(os.sep)
        if depth > 4:
            continue
        n = sum(1 for f in fn if f.lower().endswith(IMG_EXT))
        print("  " * depth, os.path.basename(dp) or ".", "(" + str(n) + " images)" if n else "")
    raise SystemExit("Edit EXPECTED in Cell 1 to match these names, then re-run.")

print("\nClass folders found at:", level)
if os.path.abspath(level) != os.path.abspath(RAW):
    for name in os.listdir(level):
        s, d = os.path.join(level, name), os.path.join(RAW, name)
        if os.path.abspath(s) != os.path.abspath(RAW) and not os.path.exists(d):
            shutil.move(s, d)

CLASSES = sorted(d for d in os.listdir(RAW) if os.path.isdir(os.path.join(RAW, d)))
print("\nCLASSES:", CLASSES)

---
## Cell 3 - Count what you really have, then split without leakage

**This is the most important cell.** It is the correction at the heart of your
resubmission, so be ready to explain it in your own words.

**The problem.** Many datasets are padded by saving rotated and zoomed copies as
separate files. If photograph 35 became thirteen files and you split those files
at random, some copies go to training and others to testing. The model learns from
a rotated copy of photograph 35, then gets tested on a zoomed copy of the same
leaf. It has already seen that leaf, that lesion, that background. The test then
measures memory, not skill. In the previous submission every single test
photograph also appeared in training.

**The fix.** Group every file by the photograph it came from, and send each whole
group to one split only.

Two columns are printed. **files** is how many image files exist. **photographs**
is how many distinct photographs they came from. Write both down.

At the end the split is verified. The overlap counts must be zero. Screenshot
that, because it is your evidence that the evaluation is sound.

In [ ]:
def base_id(fname):
    """Which original photograph did this file come from?"""
    stem = os.path.splitext(fname)[0]
    if any(h in stem.lower() for h in AUG_HINTS):
        m = re.search(r"(\d+)$", stem)
        if m:
            return m.group(1)
    return stem

# ------------- inventory -------------
groups_per_class, hashes = {}, defaultdict(list)
total_files = total_photos = 0

print("{:30s} {:>7s} {:>13s}".format("class", "files", "photographs"))
print("-" * 54)
for cls in CLASSES:
    cdir = os.path.join(RAW, cls)
    fnames = [f for f in os.listdir(cdir) if f.lower().endswith(IMG_EXT)]
    groups = defaultdict(list)
    for f in fnames:
        groups[base_id(f)].append(f)
        try:
            h = hashlib.md5(open(os.path.join(cdir, f), "rb").read()).hexdigest()
            hashes[h].append(cls + "/" + f)
        except OSError:
            pass
    groups_per_class[cls] = groups
    total_files += len(fnames)
    total_photos += len(groups)
    print("{:30s} {:7d} {:13d}".format(cls[:30], len(fnames), len(groups)))
print("-" * 54)
print("{:30s} {:7d} {:13d}".format("TOTAL", total_files, total_photos))

dups = {h: v for h, v in hashes.items() if len(v) > 1}
redundant = sum(len(v) - 1 for v in dups.values())
cross = [v for v in dups.values() if len(set(x.split("/")[0] for x in v)) > 1]
print("\nidentical duplicate files :", redundant)
print("duplicates across classes :", len(cross),
      "  <-- serious, one image labelled two ways" if cross else "")
for grp in cross[:2]:
    extra = len(grp) - 4
    print("    ", ", ".join(grp[:4]), ("... and " + str(extra) + " more") if extra > 0 else "")

if total_photos < total_files:
    print("\nOFFLINE AUGMENTATION: about {:.0f} files per photograph.".format(
          total_files / total_photos))
    print("The real size is {} photographs, not {} images.".format(total_photos, total_files))
else:
    print("\nNo offline augmentation. Each file is its own photograph.")

# ------------- grouped split -------------
if os.path.exists(SPLIT_DIR):
    shutil.rmtree(SPLIT_DIR)

plan = defaultdict(list)
print("\n{:30s} {:>6s} {:>5s} {:>5s}  (photographs)".format("class", "train", "val", "test"))
print("-" * 54)
for cls in CLASSES:
    ids = sorted(groups_per_class[cls])
    random.shuffle(ids)
    n = len(ids)
    n_tr, n_va = int(TRAIN_FRAC * n), int(0.15 * n)
    buckets = {"train": ids[:n_tr],
               "validation": ids[n_tr:n_tr + n_va],
               "test": ids[n_tr + n_va:]}
    print("{:30s} {:6d} {:5d} {:5d}".format(
        cls[:30], len(buckets["train"]), len(buckets["validation"]), len(buckets["test"])))
    for split, gids in buckets.items():
        for gid in gids:
            for f in groups_per_class[cls][gid]:
                plan[split].append((cls, f))

for split, items in plan.items():
    for cls, f in items:
        d = os.path.join(SPLIT_DIR, split, cls)
        os.makedirs(d, exist_ok=True)
        shutil.copy(os.path.join(RAW, cls, f), os.path.join(d, f))

print("\nIMAGE FILES PER SPLIT")
for split in ("train", "validation", "test"):
    n = sum(len(os.listdir(os.path.join(SPLIT_DIR, split, c))) for c in CLASSES)
    print("  {:11s} {:6d}".format(split, n))

# ------------- verification -------------
ids_in = {}
for split in ("train", "validation", "test"):
    s = set()
    for cls in CLASSES:
        for f in os.listdir(os.path.join(SPLIT_DIR, split, cls)):
            s.add((cls, base_id(f)))
    ids_in[split] = s

print("\nLEAKAGE VERIFICATION")
clean = True
for split in ("validation", "test"):
    ov = len(ids_in["train"] & ids_in[split])
    print("  photographs shared between train and {}: {} of {}".format(
        split, ov, len(ids_in[split])))
    clean = clean and ov == 0
print("\n  RESULT:", "NO LEAKAGE - safe to train and report" if clean
      else "LEAKAGE PRESENT - stop, do not report these results")

---
## Cell 4 - Load the images and weight the classes

**Augmentation happens on the fly, on training images only.** Keras varies each
image slightly as it feeds the model, so nothing extra is saved to disk and
nothing can leak. Validation and test images are never augmented, because you are
measuring performance on real images.

**Class weights** matter when one class has far more images than another. Without
them a model can score well by favouring the largest class while ignoring the
smallest. Weights make every class count equally in the loss. Above an imbalance
of about 3, report macro F1 rather than plain accuracy.

In [ ]:
train_aug = ImageDataGenerator(
    rescale=1./255, rotation_range=25, zoom_range=0.2,
    width_shift_range=0.15, height_shift_range=0.15, shear_range=0.15,
    horizontal_flip=True, vertical_flip=True,
    brightness_range=[0.8, 1.2], fill_mode="nearest")
eval_only = ImageDataGenerator(rescale=1./255)      # never augmented

train_gen = train_aug.flow_from_directory(
    SPLIT_DIR + "/train", target_size=(IMG, IMG), batch_size=BATCH,
    class_mode="categorical", classes=CLASSES, shuffle=True, seed=SEED)
val_gen = eval_only.flow_from_directory(
    SPLIT_DIR + "/validation", target_size=(IMG, IMG), batch_size=BATCH,
    class_mode="categorical", classes=CLASSES, shuffle=False)
test_gen = eval_only.flow_from_directory(
    SPLIT_DIR + "/test", target_size=(IMG, IMG), batch_size=BATCH,
    class_mode="categorical", classes=CLASSES, shuffle=False)

counts = Counter(train_gen.classes)
total = sum(counts.values())
CLASS_WEIGHT = {i: total / (len(CLASSES) * counts[i]) for i in range(len(CLASSES))}

print("\nTraining images per class, with the weight applied:")
for i, c in enumerate(CLASSES):
    print("  {:28s} {:5d}   weight {:.2f}".format(c[:28], counts[i], CLASS_WEIGHT[i]))

imbalance = max(counts.values()) / min(counts.values())
print("\nImbalance ratio: {:.1f}x".format(imbalance))
if imbalance > 3:
    print("Above 3x, so report macro F1 rather than plain accuracy.")

---
## Cell 5 - Models, and an evaluation function that does not lie

Four faults in the original evaluation code are fixed here. Each is a likely viva
question.

1. **The generator is reset before predicting.** Predicting on a generator already
   part-used starts mid-data, so predictions stop lining up with labels and the
   confusion matrix becomes meaningless.
2. **Every test image is used.** The old code used `samples // batch_size`, and
   integer division silently dropped the final partial batch.
3. **`shuffle=False` on the test generator**, so labels stay in file order.
4. **Per-class precision, recall and F1 are reported.** One accuracy figure hides a
   class the model always gets wrong.

Two models. Your own CNN is unchanged so the comparison is fair. MobileNetV2
starts from features learned from over a million images, with that base frozen so
only a small head trains. That suits a dataset this size, and the gap between the
two is your experimental finding.

In [ ]:
def baseline_cnn():
    """The project's original architecture, unchanged."""
    return models.Sequential([
        layers.Input((IMG, IMG, 3)),
        layers.Conv2D(16, 3, activation="relu"), layers.MaxPooling2D(2),
        layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation="relu"), layers.MaxPooling2D(2),
        layers.Conv2D(128, 3, activation="relu"), layers.MaxPooling2D(2),
        layers.Conv2D(256, 3, activation="relu"), layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(512, activation="relu"), layers.Dropout(0.5),
        layers.Dense(len(CLASSES), activation="softmax"),
    ], name="baseline_cnn")


def transfer_model():
    """MobileNetV2 as a frozen feature extractor plus a small head."""
    base = tf.keras.applications.MobileNetV2(
        input_shape=(IMG, IMG, 3), include_top=False, weights="imagenet")
    base.trainable = False
    return models.Sequential([
        layers.Input((IMG, IMG, 3)), base,
        layers.GlobalAveragePooling2D(),
        layers.BatchNormalization(),
        layers.Dense(128, activation="relu"), layers.Dropout(0.4),
        layers.Dense(len(CLASSES), activation="softmax"),
    ], name="mobilenetv2_transfer")


def make_callbacks():
    return [callbacks.EarlyStopping(monitor="val_loss", patience=6,
                                   restore_best_weights=True, verbose=1),
            callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3,
                                        patience=3, min_lr=1e-6, verbose=1)]


def evaluate(model, gen, name):
    gen.reset()                                            # fault 1
    steps = int(np.ceil(gen.samples / gen.batch_size))     # fault 2
    probs = model.predict(gen, steps=steps, verbose=0)[:gen.samples]
    y_pred, y_true = probs.argmax(axis=1), gen.classes[:gen.samples]

    acc = float((y_pred == y_true).mean())
    p, r, f1, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=range(len(CLASSES)), zero_division=0)

    print("\n===== " + name + " =====")
    print("test images   :", gen.samples)
    print("test accuracy : {:.4f}".format(acc))
    print("macro F1      : {:.4f}\n".format(np.mean(f1)))
    print("{:30s} {:>7s} {:>8s} {:>7s} {:>6s}".format("class", "prec", "recall", "F1", "n"))
    for i, c in enumerate(CLASSES):
        print("{:30s} {:7.3f} {:8.3f} {:7.3f} {:6d}".format(
            c[:30], p[i], r[i], f1[i], sup[i]))

    return dict(name=name, accuracy=acc, macro_f1=float(np.mean(f1)),
                per_class={c: dict(precision=float(p[i]), recall=float(r[i]),
                                   f1=float(f1[i]), support=int(sup[i]))
                           for i, c in enumerate(CLASSES)},
                confusion_matrix=confusion_matrix(
                    y_true, y_pred, labels=range(len(CLASSES))).tolist(),
                n_test=int(gen.samples))

print("Models and evaluate() ready.")

---
## Cell 6 - Experiment 1: your own CNN, trained from scratch

Watch **`val_accuracy`**, not `accuracy`. The first is performance on images held
back, which is the number that means something.

Training stops by itself when validation loss stops improving, so it may end well
before epoch 30. That is EarlyStopping working, not a fault. Screenshot the
results table.

In [ ]:
tf.keras.utils.set_random_seed(SEED)
m1 = baseline_cnn()
m1.summary()

m1.compile(optimizer=optimizers.Adam(LR),
           loss="categorical_crossentropy", metrics=["accuracy"])
hist1 = m1.fit(train_gen, validation_data=val_gen, epochs=EPOCHS,
               callbacks=make_callbacks(), class_weight=CLASS_WEIGHT, verbose=1)

res1 = evaluate(m1, test_gen, "Baseline CNN (from scratch)")

---
## Cell 7 - Experiment 2: MobileNetV2 transfer learning

Note the gap between total and trainable parameters. Only the small head trains,
which is why this works on a modest dataset where fitting millions of weights from
scratch does not.

In [ ]:
tf.keras.utils.set_random_seed(SEED)
m2 = transfer_model()
print("total parameters     : {:,}".format(m2.count_params()))
print("trainable parameters : {:,}\n".format(
    sum(int(tf.size(w)) for w in m2.trainable_weights)))

m2.compile(optimizer=optimizers.Adam(LR),
           loss="categorical_crossentropy", metrics=["accuracy"])
hist2 = m2.fit(train_gen, validation_data=val_gen, epochs=EPOCHS,
               callbacks=make_callbacks(), class_weight=CLASS_WEIGHT, verbose=1)

res2 = evaluate(m2, test_gen, "MobileNetV2 (transfer learning)")

---
## Cell 8 - Figures, tables, saved model and download

Three figures at 200 dpi, sharp enough to print. A marker called your graphics
poor quality and taken from existing literature, so these must be your own. Number
and caption each one, and refer to each in the text.

**Reading a confusion matrix.** Rows are the true class, columns the guess. The
diagonal is correct. Find the largest off-diagonal number and name the two classes
it joins in your discussion. That one sentence shows you examined your own results
instead of quoting a headline figure.

The download at the end holds everything: figures, raw metrics and the model.

In [ ]:
RUNS = [("Baseline CNN\n(from scratch)", hist1.history, res1, m1, "baseline_cnn"),
        ("MobileNetV2\n(transfer learning)", hist2.history, res2, m2,
         "mobilenetv2_transfer")]

for label, h, r, model, tag in RUNS:
    json.dump({k: [float(x) for x in v] for k, v in h.items()},
              open("results/" + tag + "_history.json", "w"), indent=2)
    json.dump(r, open("results/" + tag + "_metrics.json", "w"), indent=2)

# ---------- Figure 1: curves ----------
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
for j, (label, h, r, _, _) in enumerate(RUNS):
    ep = range(1, len(h["accuracy"]) + 1)
    axes[0, j].plot(ep, h["accuracy"], "o-", ms=3, c="#c0392b", label="train")
    axes[0, j].plot(ep, h["val_accuracy"], "s-", ms=3, c="#2471a3", label="validation")
    axes[0, j].set_title(label, fontweight="bold")
    axes[0, j].set_xlabel("Epoch")
    axes[0, j].set_ylabel("Accuracy")
    axes[0, j].set_ylim(0, 1.02)
    axes[0, j].legend(fontsize=8)
    axes[1, j].plot(ep, h["loss"], "o-", ms=3, c="#c0392b", label="train")
    axes[1, j].plot(ep, h["val_loss"], "s-", ms=3, c="#2471a3", label="validation")
    axes[1, j].set_xlabel("Epoch")
    axes[1, j].set_ylabel("Loss")
    axes[1, j].legend(fontsize=8)
fig.suptitle("Training and validation curves", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig("figures/fig1_training_curves.png", bbox_inches="tight")
plt.show()

# ---------- Figure 2: confusion matrices ----------
side = max(6.0, 1.0 * len(CLASSES))
fig, axes = plt.subplots(1, 2, figsize=(2 * side, side * 0.85))
for j, (label, h, r, _, _) in enumerate(RUNS):
    sns.heatmap(np.array(r["confusion_matrix"]), annot=True, fmt="d",
                cmap="Blues", cbar=False, xticklabels=CLASSES,
                yticklabels=CLASSES, ax=axes[j], annot_kws={"size": 9})
    axes[j].set_title(label.replace("\n", " ") +
                      "\naccuracy {:.3f}  |  macro F1 {:.3f}".format(
                          r["accuracy"], r["macro_f1"]),
                      fontweight="bold", fontsize=10)
    axes[j].set_xlabel("Predicted")
    axes[j].set_ylabel("Actual")
    axes[j].tick_params(axis="x", rotation=45)
    axes[j].tick_params(axis="y", rotation=0)
fig.suptitle("Confusion matrices on the held-out test set",
             fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig("figures/fig2_confusion_matrices.png", bbox_inches="tight")
plt.show()

# ---------- Figure 3: comparison ----------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
short = [r[0] for r in RUNS]
cols = ["#e67e22", "#1e8449"]
acc = [r[2]["accuracy"] for r in RUNS]
f1s = [r[2]["macro_f1"] for r in RUNS]
x = np.arange(2)
bw = 0.35
bars_a = ax1.bar(x - bw / 2, acc, bw, color=cols, alpha=0.95, label="Test accuracy")
bars_f = ax1.bar(x + bw / 2, f1s, bw, color=cols, alpha=0.55, label="Macro F1")
for b in list(bars_a) + list(bars_f):
    ax1.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.015,
             "{:.3f}".format(b.get_height()), ha="center",
             fontsize=9, fontweight="bold")
chance = 1.0 / len(CLASSES)
ax1.axhline(chance, ls="--", c="grey", lw=1)
ax1.text(1.45, chance + 0.02, "random guess ({:.0f}%)".format(chance * 100),
         fontsize=8, c="grey", ha="right")
ax1.set_xticks(x)
ax1.set_xticklabels(short, fontsize=9)
ax1.set_ylabel("Score")
ax1.set_ylim(0, 1.12)
ax1.set_title("Headline test performance", fontweight="bold")
ax1.legend(fontsize=8, loc="upper left")

xc = np.arange(len(CLASSES))
bw2 = 0.35
for j, (label, h, r, _, _) in enumerate(RUNS):
    ax2.bar(xc + (j - 0.5) * bw2, [r["per_class"][c]["f1"] for c in CLASSES],
            bw2, color=cols[j], alpha=0.9, label=label.replace("\n", " "))
ax2.set_xticks(xc)
ax2.set_xticklabels([c.replace(" ", "\n") for c in CLASSES], fontsize=8)
ax2.set_ylabel("F1 score")
ax2.set_ylim(0, 1.12)
ax2.set_title("Per-class F1 score", fontweight="bold")
ax2.legend(fontsize=8)
fig.tight_layout()
fig.savefig("figures/fig3_model_comparison.png", bbox_inches="tight")
plt.show()

# ---------- results table ----------
rows = ["| Model | Parameters | Trainable | Epochs | Test images | Test accuracy | Macro F1 |",
        "|---|---|---|---|---|---|---|"]
for label, h, r, model, _ in RUNS:
    tr = sum(int(tf.size(w)) for w in model.trainable_weights)
    rows.append("| {} | {:,} | {:,} | {} | {} | {:.4f} | {:.4f} |".format(
        label.replace("\n", " "), model.count_params(), tr,
        len(h["accuracy"]), r["n_test"], r["accuracy"], r["macro_f1"]))
table = "\n".join(rows)
open("figures/results_table.md", "w").write(table)
print(table)

best = max(RUNS, key=lambda t: t[2]["macro_f1"])
print("\n\nPER-CLASS TABLE - best model:", best[2]["name"])
print("| Class | Precision | Recall | F1 | Test images |")
print("|---|---|---|---|---|")
for c in CLASSES:
    d = best[2]["per_class"][c]
    print("| {} | {:.3f} | {:.3f} | {:.3f} | {} |".format(
        c, d["precision"], d["recall"], d["f1"], d["support"]))

# ---------- save and download ----------
best[3].save("models/cotton_model.keras")
json.dump(CLASSES, open("models/class_names.json", "w"), indent=2)
print("\nSaved models/cotton_model.keras ({:.1f} MB)".format(
    os.path.getsize("models/cotton_model.keras") / 1e6))

!zip -qr training_output.zip figures results models
files.download("training_output.zip")

---
## Prepare for the viva

The last attempt failed partly because these went unanswered. Work through them
using your own output above, and write the answers down.

**Data**

1. How many original photographs per class, and how many image files? Why do those
   numbers differ?
2. What is data leakage, and how did you show your split was free of it? Quote the
   verification numbers from Cell 3.
3. Why are validation and test images never augmented?
4. Why did you use class weights, and what is your imbalance ratio?

**Models**

5. Why does transfer learning suit this dataset better than training from scratch?
   Give the parameter counts.
6. What does freezing the MobileNetV2 base mean, and what would happen if you
   unfroze it with this much data?
7. What does EarlyStopping do, and which epoch did it restore for each model?

**Results**

8. State your test accuracy and macro F1. Why report both?
9. Which class performs worst, and which is it most often confused with? Point at
   the cell in Figure 2.
10. Why is the accuracy less precise than the test image count suggests?
11. What would you do next, and what do you expect it to improve?

**System**

12. Why must `IMG_SIZE` in `app.py` equal `IMG` here?
13. Walk through what happens between a user uploading a leaf and the prediction
    appearing.

---

### After this notebook

1. Put `figures/` and `results/` into your GitHub repository
2. Keep `cotton_model.keras` on Google Drive and link it in the README
3. Copy `class_names.json` into your local `models/` folder
4. Run the web app, upload a leaf, screenshot the working prediction
5. Write the Experimental Modelling section around Figures 1 to 3 and the tables